*Load relevant packages:*

In [1]:
clean_up = True # if True, remove all gams related files from working folder before starting
%run packages.ipynb
# Local packages:
os.chdir(d['py'])
import mCGE 
os.chdir(os.path.join(d['curr'], 'py'))
import report

# Compute embodied materials

The notebook loads the calibrated model and computes embodied material intensities as a simple post-solution routine. 

*Settings:*

In [2]:
t0 = 2019 # baseline year
v = 'vMain' # global name
name = f'{v}{t0}CGE' # name used when storing calibrated model instance

*Load:*

In [3]:
M = mCGE.WasteManagementCGE.load(os.path.join(d['data'], name)) # load model
ws = M.ws 
db0 = M.db.copy() # store initial solution

*Initialize reporting class:*

In [4]:
Rep = report.Standard(db0)

### CHEK STUFF:

Get parameter values:

In [5]:
db0('qD').xs(t0).xs('RxE_Plastic_V',level='n')

s
other          1.184817
man_oth        3.254573
man_tex        0.687094
man_rub_pla    5.316694
man_met        0.600709
Energy         0.015233
I_K            0.080875
Name: qD, dtype: float64

In [6]:
db0('qD').xs(t0).xs('RxE_Plastic_R',level='n')

s
man_oth        0.011753
man_rub_pla    0.184980
Name: qD, dtype: float64

In [7]:
db0('mu').xs('RxE_Plastic',level='n')

s            nn           
man_oth      RxE_Plastic_V    0.996454
             RxE_Plastic_R    0.003546
man_rub_pla  RxE_Plastic_V    0.966081
             RxE_Plastic_R    0.033919
Name: mu, dtype: float64

$$\begin{align}
\frac{\partial Y/\partial Q_v}{\partial Y/\partial Q_R} = \left(\frac{\mu_V}{\mu_R}\right)^{1/\sigma} \left(\frac{q_R}{q_V}\right)^{\frac{1}{\sigma}} \approx 4.6 ... 
\end{align}$$

### CHEK STUFF:

For each industry $i$, we define the net contribution of materials *not including embodied materials* from the definition:

$$
\begin{align}
    X_{i,m} = \text{Ext}_m+Y_{m,i}^D+ Y_{m,i}^F - Y_{m,i}^S - W_{m,i} - M_{m,i}, \tag{1}
\end{align}
$$

where $\text{Ext}_{i,m}$ is extraction, $Y_{m,i}^D+ Y_{m,i}^F$ is the consumption of virgin/recycled materials from domestic/foreign origin, $Y_{m,i}^s$ is the supply of materials virgin/recycled, $W_{m,i}$ are waste materials, and $M_{m,i}$ are emissions to the environment. For our purposes, we note:
* We have no explicit extraction,
* We have no explicit emissions/balance category. *To do: add balance category here to see the effect on embodied emissions.* 

*Define industries' total supply and demand of materials:*

In [8]:
supplyM = adjMultiIndex.applyMult(adj.rc_pd(db0('qS'), Rep.vm.union(Rep.rm)), Rep.n2m).groupby(['t','s','m']).sum()
demandM = adjMultiIndex.applyMult(adj.rc_pd(db0('qD'), Rep.vm.union(Rep.rm)), Rep.n2m).groupby(['t','s','m']).sum()

*Define $X_{i,m}$ as demand minus supply and waste materials:*

In [9]:
X = demandM.add(-supplyM, fill_value=0).add(-db0('qWS'),fill_value=0)

Next, define the net supply of each final good in each industry (values or quantities, not important here):

In [10]:
dom2for_services = adj.rc_pd(db0('dom2for'), db0('n_p')).rename(['nn','n']) # mapping 
for_services = dom2for_services.get_level_values('n').unique() # foreign services 
# Get imported services mapped to the domestic name (e.g. 'Energy_F' --> 'Energy')
importServices = adjMultiIndex.applyMult(adj.rc_pd(db0('qD'), for_services), dom2for_services).droplevel('n').rename_axis(index = {'nn':'n'})

Write up supply of services minus demand (where imports of type $n_F$ is mapped to the domestic name $n$):

In [11]:
supplyS = adj.rc_pd(db0('qS'), db0('n_p').union(db0('inv_p')))
demandS = adj.rc_pd(db0('qD'), db0('n_p').union(db0('inv_p'))).add(importServices, fill_value=0)
netSupply = supplyS.add(-demandS, fill_value=0)

Look at Danish industries only:

In [12]:
netSupply = adj.rc_pd(netSupply, db0('s_p').union(db0('s_i')))
X = adj.rc_pd(X, db0('s_p').union(db0('s_i')))

### Look at embodied materials for a single year:

Get levels for a single year, sort, and unstack to get A matrix with industries `s` in the rows and goods `n` in the columns:

In [13]:
Xt = X.xs(t0).sort_index()
At = netSupply.xs(t0).sort_index().unstack('n')

Invert matrix:

In [14]:
Ainv = pd.DataFrame(np.linalg.inv(At.values), index = At.columns, columns = At.index)

The `Ainv` matrix has to be repeated for all $m$. However, not all industry/material/year combinations are in `Xt` vector. There are different ways of making sure this works, here we repeat the `Ainv` matrix for all relevant `Xt[s,m]`, then multiply the two and sum over $s$.

In [15]:
θ = (adjMultiIndex.bc(Ainv.stack(), Xt) * Xt).groupby(['n','m']).sum().sort_values()

A quick check that this is the same as simple matrix multiplication for one material type:

In [118]:
simpleMethod = pd.Series(Ainv.values @ Xt.xs('Metal',level='m').values, index = Xt.xs('Metal',level='m').index).sort_values()
assert abs(simpleMethod-θ.xs('Metal',level='m')).max()<1e-9, "Check vectorized method for calculating θ" 

Add some arbitrary level and check:

In [167]:
Xtest = Xt+5
θtest = (adjMultiIndex.bc(Ainv.stack(), Xtest) * Xtest).groupby(['n','m']).sum().sort_values()
θtest

n            m      
sale         Textile    0.017237
pub          Metal      0.020736
other        Metal      0.020758
pub          Textile    0.022451
other        Textile    0.024494
pub          Paper      0.025486
constr       Textile    0.026273
pub          Rubber     0.026715
sale         Metal      0.028077
             Paper      0.028229
pub          Plastic    0.029317
other        Paper      0.029957
Energy       Textile    0.030512
other        Rubber     0.031357
sale         Rubber     0.031668
other        Plastic    0.035471
sale         Plastic    0.037451
I_K          Textile    0.037955
constr       Metal      0.038055
man_oth      Textile    0.039723
             Paper      0.045581
I_K          Metal      0.045751
man_oth      Rubber     0.047725
             Metal      0.049423
constr       Paper      0.049787
             Rubber     0.050173
I_K          Paper      0.050304
             Rubber     0.051344
constr       Plastic    0.056590
I_K          Plastic  